## 基本関数定義、データ生成部分

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cupy as cp

N = 40
F = 8.0

def L96(x,t,F = 8):
  return (cp.roll(x,-1,axis = 0) - cp.roll(x,2, axis = 0)) * cp.roll(x,1,axis = 0) - x + F

def rk4(f, x0, t, F=8.0):
    # len(x0) ではなく *x0.shape を使うことで、1次元でも2次元でも柔軟にサイズを合わせる
    x = cp.zeros((len(t), *x0.shape))
    x[0] = x0
    for i in range(1, len(t)):
        dt = t[i] - t[i-1]
        k1 = f(x[i-1], t[i-1], F)
        k2 = f(x[i-1] + 0.5 * dt * k1, t[i-1] + 0.5 * dt, F)
        k3 = f(x[i-1] + 0.5 * dt * k2, t[i-1] + 0.5 * dt, F)
        k4 = f(x[i-1] + dt * k3, t[i-1] + dt, F)
        x[i] = x[i-1] + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)
    return x

def gaspari_cohn(d, c):
    """
    距離 d とカットオフパラメータ c に基づくGaspari-Cohn関数の重みを計算
    スライド23ページに準拠した5次スプライン関数
    """
    r = abs(d) / c
    if r <= 1.0:
        return 1.0 - (5.0/3.0)*r**2 + (5.0/8.0)*r**3 + (1.0/2.0)*r**4 - (1.0/4.0)*r**5
    elif r <= 2.0:
        return 4.0 - 5.0*r + (5.0/3.0)*r**2 + (5.0/8.0)*r**3 - (1.0/2.0)*r**4 + (1.0/12.0)*r**5 - (2.0/3.0)*(1.0/r)
    else:
        return 0.0

def get_localization_matrix(N, sigma):
    L = cp.zeros((N, N))
    c = cp.sqrt(10.0/3.0) * sigma  # スライド p.23 の定義
    
    for i in range(N):
        for j in range(N):
            # Lorenz-96はリング状なので、左右どちら回りでの最短距離かを判定
            dist = min(abs(i - j), N - abs(i - j))
            L[i, j] = gaspari_cohn(dist, c)
            
    return L

L_matrix = get_localization_matrix(N, sigma=5.0)


rng = cp.random.default_rng(seed=42)

## 真値と観測データの生成
x_minus90 = cp.full(N,F)
x_minus90[19] = F + 1.001
spinup_days = 360
t_spinup = cp.arange(0.0,spinup_days / 5,0.05)
x_spinup = rk4(L96, x_minus90, t_spinup, F=F)
t_truth = cp.arange(0.0,spinup_days / 5,0.05)
x_truth = rk4(L96, x_spinup[-1], t_truth, F=F)
#y_observe = x_truth + rng.standard_normal(size=x_truth.shape)

H = cp.eye(N)
R = cp.eye(N) * 1.0
R_inv = cp.linalg.inv(R)
num_obs = N
ensembles = 10
sigma_loc = 4
inflation = 0.05
ensemble_init = cp.zeros((N,ensembles))

y_observe = x_truth @ H.T + rng.standard_normal(size=x_truth.shape)

noise = rng.standard_normal(size=(N,ensembles))
noise = noise - cp.mean(noise, axis=1, keepdims=True)  #ノイズの平均が厳密に0になるようにしとく
for i in range(ensembles):
  member_x = x_spinup[-1] + noise[:, i]
  member_x_spinup = rk4(L96, member_x, t_spinup, F=F)
  ensemble_init[:, i] = member_x_spinup[-1]

In [ ]:
import cupy as cp
import seaborn as sns

# =====================================================================
# 【パラメータ設定】Miyoshi (2011) 適応共分散膨張用
# =====================================================================
# 論文の共分散膨張パラメータ α の初期値（例: 従来の (1+inflation)**2 に相当）
alpha_init = (1.0 + inflation) ** 2  # または初期値 1.05 など
curr_alpha = cp.full(N, alpha_init)

# 事前分散パラメータ v^b (論文 Section 5 より固定のチューニングパラメータ)
# 論文内の実験では v^b = 0.01^2 ~ 0.04^2 程度が推奨・使用されています
v_b = 0.02**2
# =====================================================================

i = 0
history_X_analysis = [ensemble_init.copy()]
history_alpha = [curr_alpha.copy()]  # 各時刻のインフレーション推移を保存するリスト
curr_X_analysis = ensemble_init.copy()

for step in range(1, len(t_truth)):
    t_step = t_truth[step - 1 : step + 1]

    curr_X_background = rk4(L96, curr_X_analysis, t_step, F=F)[-1]
    x_background_mean = cp.mean(curr_X_background, axis=1)

    # 1. 膨張前の生の背景アンサンブル摂動 Z_raw と Y_raw を計算
    # (適応インフレーションの推定値 α^o を正しく導出するために必要)
    Z_raw = (curr_X_background - x_background_mean[:, None]) / cp.sqrt(
        ensembles - 1
    )
    Y_raw = H @ Z_raw

    # 2. 背景アンサンブル摂動に共分散膨張パラメータ α の「平方根」を掛ける
    Z_background = Z_raw * cp.sqrt(curr_alpha[:, None])
    Y_background = H @ Z_background

    d_ob = y_observe[step] - H @ x_background_mean
    next_X_analysis = cp.zeros_like(curr_X_analysis)
    next_alpha = cp.zeros_like(curr_alpha)

    # (2) 局所解析ステップ: 各格子点(変数)ごとのループ
    for j in range(N):

        # 2.1 局所化 (Localization)
        dist = cp.minimum(cp.abs(cp.arange(N) - j), N - cp.abs(cp.arange(N) - j))
        loc_weight = cp.exp(-(dist**2) / (2 * sigma_loc**2))
        R_loc_inv = cp.diag(loc_weight) @ R_inv

        # -----------------------------------------------------------------
        # 【動的共分散膨張の更新】Miyoshi (2011) Section 5
        # -----------------------------------------------------------------
        # 式(14): イノベーション統計から推定される観測インフレーション α^o_j
        # 巡回性より, d@d^t@R と同じ
        do_R_do = d_ob.T @ R_loc_inv @ d_ob
        tr_rho = cp.sum(loc_weight)

        # 分母 tr(H P^f H^T * rho * R^-1) ※P^fは膨張前のサンプル共分散
        # 巡回性とY = HZ から数式書き換えれる（ノートに書いて確認済み）
        tr_HPH = cp.trace(Y_raw.T @ R_loc_inv @ Y_raw)

        alpha_o = (do_R_do - tr_rho) / tr_HPH

        # 式(15): 中心極限定理から導かれる推定値 α^o_j の分散 v^o_j
        v_o = (2.0 / tr_rho) * (curr_alpha[j] + tr_rho / tr_HPH) ** 2

        # 式(6): カルマンフィルタ解析更新による事後インフレーション α^a_j
        k_gain = v_b / (v_b + v_o)
        next_alpha[j] = curr_alpha[j] + k_gain * (alpha_o - curr_alpha[j])

        # 数値的安定性のための下限クリッピング (実用上の安全策)
        next_alpha[j] = cp.maximum(1.0, next_alpha[j])
        # -----------------------------------------------------------------

        # 2.2 アンサンブル空間での解析誤差共分散 (P~a) の逆行列を計算
        Pa_inv = cp.eye(ensembles) + Y_background.T @ R_loc_inv @ Y_background

        # 2.3 固有値分解 (EVD) による平方根行列の計算
        eigenvalues, eigenvectors = cp.linalg.eigh(Pa_inv)

        Pa = eigenvectors @ cp.diag(1.0 / eigenvalues) @ eigenvectors.T
        Pa_sqrt = (
            eigenvectors @ cp.diag(1.0 / cp.sqrt(eigenvalues)) @ eigenvectors.T
        )

        # 2.4 変換行列 T の計算
        w_mean = Pa @ Y_background.T @ R_loc_inv @ d_ob
        W_pert = cp.sqrt(ensembles - 1) * Pa_sqrt

        T_matrix = cp.outer(w_mean, cp.ones(ensembles)) + W_pert

        # 2.5 アンサンブルの更新 (着目している格子点 j のみ更新)
        next_X_analysis[j, :] = (
            x_background_mean[j] + Z_background[j, :] @ T_matrix
        )

        if i == 0:
            print("dist.shape:", dist.shape)
            print("loc_weight.shape:", loc_weight.shape)
            print("R_loc_inv.shape:", R_loc_inv.shape)
            print("Pa_inv.shape:", Pa_inv.shape)
            print("Pa.shape:", Pa.shape)
            print("Pa_sqrt.shape:", Pa_sqrt.shape)
            print("w_mean.shape:", w_mean.shape)
            print("W_pert.shape:", W_pert.shape)
            print("T_matrix.shape:", T_matrix.shape)
            i += 1

    # 解析結果とインフレーションパラメータを更新して保存
    curr_X_analysis = next_X_analysis
    history_X_analysis.append(curr_X_analysis.copy())

    curr_alpha = next_alpha
    history_alpha.append(curr_alpha.copy())

final_analysis_trajectory = cp.mean(cp.array(history_X_analysis), axis=2)

dist.shape: (40,)
loc_weight.shape: (40,)
R_loc_inv.shape: (40, 40)
Pa_inv.shape: (10, 10)
Pa.shape: (10, 10)
Pa_sqrt.shape: (10, 10)
w_mean.shape: (10,)
W_pert.shape: (10, 10)
T_matrix.shape: (10, 10)


In [6]:
import cupy as cp

# =====================================================================
# 【追加の事前準備】適応共分散膨張用のパラメータ・初期化
# =====================================================================
# α（背景誤差共分散の膨張倍率）の初期値
alpha_init = (1.0 + inflation) ** 2
curr_alpha = cp.full(N, alpha_init)

# 事前分散パラメータ v^b (時系列更新の滑らかさを決める固定チューニング値)
# Miyoshi(2011) の実験準拠: 0.01^2 ~ 0.04^2 程度が推奨
v_b = 0.02**2

# =====================================================================

history_X_analysis = [ensemble_init.copy()]
history_alpha = [curr_alpha.copy()]  # 各時刻のインフレーション推移
curr_X_analysis = ensemble_init.copy()

# === ループ前の準備（事前計算） ===
# 1. 純粋な局所化重み行列 (サイズ: N x N)
loc_weight_all = cp.zeros((N, N))
for j in range(N):
    dist = cp.minimum(cp.abs(cp.arange(N) - j), N - cp.abs(cp.arange(N) - j))
    loc_weight_all[j] = cp.exp(-(dist**2) / (2 * sigma_loc**2))

# 【追加】各格子点における「実質的な観測データ数」 tr(rho_j) を事前計算 (サイズ: N)
tr_rho_all = cp.sum(loc_weight_all, axis=1)

# 2. 観測誤差 R_inv の対角成分を掛けた最終重み行列 (サイズ: N x N)
R_inv_diag = cp.diag(R_inv)
weight_matrix = loc_weight_all * R_inv_diag[None, :]
# =================================

for step in range(1, len(t_truth)):
    t_step = t_truth[step - 1 : step + 1]
    curr_X_background = rk4(L96, curr_X_analysis, t_step, F=F)[-1]

    x_background_mean = cp.mean(curr_X_background, axis=1)

    # 1. 生の背景アンサンブル摂動 Z_raw と Y_raw を導出 (インフレーション推定に必須)
    Z_raw = (curr_X_background - x_background_mean[:, None]) / cp.sqrt(
        ensembles - 1
    )
    Y_raw = H @ Z_raw

    d_ob = y_observe[step] - H @ x_background_mean

    # ----------------------------------------------------
    # 【動的共分散膨張のバッチ更新】Miyoshi (2011) Section 5
    # ----------------------------------------------------
    # 式(14): イノベーション統計に基づく観測インフレーション推定値 α^o (サイズ: N)
    do_R_do = cp.einsum("ji, i -> j", weight_matrix, d_ob**2)
    tr_HPH = cp.einsum("ji, im -> j", weight_matrix, Y_raw**2)

    alpha_o = (do_R_do - tr_rho_all) / tr_HPH

    # 式(15): 推定値 α^o の分散 v^o (サイズ: N)
    v_o = (2.0 / tr_rho_all) * (curr_alpha + tr_rho_all / tr_HPH) ** 2

    # 式(6): カルマンフィルタ解析更新による事後インフレーションの計算
    k_gain = v_b / (v_b + v_o)
    curr_alpha = curr_alpha + k_gain * (alpha_o - curr_alpha)

    # 数値的安定性（丸め誤差による発散防止）のための下限クリッピング
    curr_alpha = cp.maximum(1.0, curr_alpha)
    history_alpha.append(curr_alpha.copy())
    # ----------------------------------------------------

    # 2. 推定されたαの「平方根」を摂動に掛けて膨張させる [サイズブロードキャスト]
    Z_background = Z_raw * cp.sqrt(curr_alpha[:, None])
    Y_background = H @ Z_background

    # ----------------------------------------------------
    # (2) 局所解析ステップ: 完全バッチ処理
    # ----------------------------------------------------
    Pa_inv = cp.eye(ensembles)[None, :, :] + cp.einsum(
        "ji, im, in -> jmn", weight_matrix, Y_background, Y_background
    )

    eigenvalues, eigenvectors = cp.linalg.eigh(Pa_inv)

    D_inv = 1.0 / eigenvalues
    D_inv_sqrt = 1.0 / cp.sqrt(eigenvalues)
    Pa = eigenvectors @ (D_inv[..., None] * cp.swapaxes(eigenvectors, 1, 2))
    Pa_sqrt = eigenvectors @ (
        D_inv_sqrt[..., None] * cp.swapaxes(eigenvectors, 1, 2)
    )

    weighted_dob = weight_matrix * d_ob[None, :]

    Y_weighted_dob = cp.einsum("im, ji -> jm", Y_background, weighted_dob)
    w_mean = cp.einsum("jmn, jn -> jm", Pa, Y_weighted_dob)

    W_pert = cp.sqrt(ensembles - 1) * Pa_sqrt

    T_matrix = w_mean[:, :, None] + W_pert

    next_X_analysis = x_background_mean[:, None] + cp.einsum(
        "jm, jmn -> jn", Z_background, T_matrix
    )

    curr_X_analysis = next_X_analysis
    history_X_analysis.append(curr_X_analysis.copy())


final_analysis_trajectory = cp.mean(cp.array(history_X_analysis), axis=2)

In [10]:
# 1. アンサンブル平均をとって解析値の軌跡を作成 (サイズ: [Time, N])
final_analysis_trajectory = cp.mean(cp.array(history_X_analysis), axis=2)

# 2. スピンアップ期間（最初の10%）を除外
skip = int(len(t_truth) * 0.1)
stable_trajectory = final_analysis_trajectory[skip:]
stable_truth = x_truth[skip:]

# 3. 各時刻(step)ごとに、空間(N=40)の二乗平均をとってルートを被せる (サイズ: [Time - skip])
# axis=1 を指定することで、時刻ごとのRMSEの配列になります
step_rmse = cp.sqrt(cp.mean((stable_trajectory - stable_truth)**2, axis=1))

# 4. 時刻ごとのRMSEを時間平均する
rmse_inline = float(cp.mean(step_rmse))

print(f"RMSE (Inline, Fixed): {rmse_inline:.4f}")


RMSE (Inline, Fixed): 0.2164
